In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('ggplot')

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from scipy.stats import skew
from sklearn.decomposition import PCA, KernelPCA
from xgboost import XGBRegressor

In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.linear_model import ElasticNet, SGDRegressor, BayesianRidge
from sklearn.kernel_ridge import KernelRidge
from xgboost import XGBRegressor

In [ ]:
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',1000)

In [ ]:
train=pd.read_csv('./X_train.csv')
test=pd.read_csv('./X_test.csv')
price_res=pd.read_csv('./y_train.csv')
 

# data preprocessing

In [ ]:
# train .columns
# for col in train.columns:
#     print(col,train[col].dtype)

In [ ]:
# full = train

### deal with years

In [ ]:
# # Convert '建築完成年月' to datetime format
# full['建築完成年月'] = pd.to_datetime(full['建築完成年月'])

# # Get the current date
# today = datetime.today()

# df = pd.DataFrame()


# df['date_str'] = full['交易年'].astype(str) + '-' + full['交易月'].astype(str).str.zfill(2) + '-' + full['交易日'].astype(str).str.zfill(2)
# df['交易日期'] = pd.to_datetime(df['date_str'])
# #ˋ計算交易時屋齡
# full['建築年紀']=df['交易日期'].dt.year- full['建築完成年月'].dt.year


# print(full['建築年紀'])
# #
# full['交易距今時間'] = ((today.year - df['交易日期'].dt.year) * 12) + (today.month - df['交易日期'].dt.month)

# print(full['交易距今時間'])


#### drop original trade and build date

In [ ]:
# full = full.drop(['交易年', '交易日', '交易月','建築完成年月'], axis=1)

### encode

In [ ]:
# from sklearn.preprocessing import LabelEncoder

# labelencoder = LabelEncoder()

# for col in full.columns:
#     if full[col].dtype=='object':
#         full[col] = labelencoder.fit_transform(train[col])


In [ ]:
# import pandas as pd
# from datetime import datetime
# from sklearn.pipeline import Pipeline, FeatureUnion
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import FunctionTransformer, LabelEncoder
# from sklearn.base import BaseEstimator, TransformerMixin

# class DateTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self):
#         pass

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         df = X.copy()
#         df['建築完成年月'] = pd.to_datetime(df['建築完成年月'])
#         today = datetime.today()
#         df['date_str'] = df['交易年'].astype(str) + '-' + df['交易月'].astype(str).str.zfill(2) + '-' + df['交易日'].astype(str).str.zfill(2)
#         df['交易日期'] = pd.to_datetime(df['date_str'])
#         df['建築年紀'] = df['交易日期'].dt.year - df['建築完成年月'].dt.year
#         df['交易距今時間'] = ((today.year - df['交易日期'].dt.year) * 12) + (today.month - df['交易日期'].dt.month)
#         df = df.drop(['交易年', '交易月', '交易日', '建築完成年月', 'date_str', '交易日期'], axis=1)
#         return df

# class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self):
#         self.label_encoders = {}

#     def fit(self, X, y=None):
#         for col in X.select_dtypes(include=['object']).columns:
#             le = LabelEncoder()
#             le.fit(X[col])
#             self.label_encoders[col] = le
#         return self

#     def transform(self, X):
#         df = X.copy()
#         for col, le in self.label_encoders.items():
#             df[col] = le.transform(df[col])
#         return df

# # Create the pipeline
# pipeline = Pipeline([
#     ('date_transformer', DateTransformer()),
#     ('label_encoder', LabelEncoderTransformer())
# ])

# # Apply the pipeline to your data
# full = train.copy()
# full_transformed = pipeline.fit_transform(full)

# print(full_transformed)


# pipeline

In [ ]:
import pandas as pd
from datetime import datetime
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns)

class DateTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['建築完成年月'] = pd.to_datetime(df['建築完成年月'])
        today = datetime.today()
        df['date_str'] = df['交易年'].astype(str) + '-' + df['交易月'].astype(str).str.zfill(2) + '-' + df['交易日'].astype(str).str.zfill(2)
        df['交易日期'] = pd.to_datetime(df['date_str'])
        df['建築年紀'] = df['交易日期'].dt.year - df['建築完成年月'].dt.year
        df['交易距今時間'] = ((today.year - df['交易日期'].dt.year) * 12) + (today.month - df['交易日期'].dt.month)
        df = df.drop(['交易年', '交易月', '交易日', '建築完成年月', 'date_str', '交易日期'], axis=1)
        return df

class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.label_encoders = {}

    def fit(self, X, y=None):
        for col in X.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.label_encoders[col] = le
        return self

    def transform(self, X):
        df = X.copy()
        for col, le in self.label_encoders.items():
            df[col] = le.transform(df[col])
        return df

# Create the pipeline
pipeline = Pipeline([
    ('drop_columns', DropColumns(columns=[ 'Id'])),
    ('date_transformer', DateTransformer()),
    ('label_encoder', LabelEncoderTransformer())
])

# Apply the pipeline to your data
full = train.copy()
full_transformed = pipeline.fit_transform(full)
 
print(full_transformed)


In [ ]:
full_transformed.head(5)

In [ ]:
full_transformed.columns,len(full_transformed.columns)

# feature select

## corr 

In [ ]:
features_pre_drop =  [  '托兒所', '國中', '高中職', '大學',  '大賣場', '超市', '百貨公司']
 


In [ ]:
import pandas as pd
df1 = price_res
df2 = full_transformed
correlation_series = df2.corrwith(df1['單價元平方公尺'])

correlation_series = correlation_series.sort_values(ascending=False)
print(correlation_series)

In [ ]:
# Set a threshold (adjust as needed)
threshold =0.019
# Identify features to drop
print(features_pre_drop)
features_to_drop =features_pre_drop+ correlation_series[correlation_series.abs() < threshold].index.tolist()
print(features_to_drop)
# Drop the features
full_selected = full_transformed.drop(features_to_drop, axis=1)


In [ ]:
len(full_selected.columns),len(full_transformed.columns)

# model & Evaluate

In [ ]:
from sklearn .model_selection import KFold

In [ ]:
# define cross validation strategy
def rmse_cv(model,X,y):
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=5))
    return rmse


In [ ]:
models = [RandomForestRegressor(n_jobs=-1),
          ExtraTreesRegressor(n_jobs=-1),XGBRegressor()]

In [ ]:
names = [ "RF","Extra","XGB"]
for name, model in zip(names, models):
    score = rmse_cv(model,full_selected, price_res['單價元平方公尺'])
    print("{}: {:.6f}, {:.4f}".format(name,score.mean(),score.std()))

In [ ]:
names = [ "RF","Extra","XGB"]
for name, model in zip(names, models):
    score = rmse_cv(model,full_transformed, price_res['單價元平方公尺'])
    print("{}: {:.6f}, {:.4f}".format(name,score.mean(),score.std()))

In [ ]:
class grid():
    def __init__(self,model):
        self.model = model
    
    def grid_get(self,X,y,param_grid):
        grid_search = GridSearchCV(self.model,param_grid,cv=5, scoring="neg_mean_squared_error")
        grid_search.fit(X,y)
        print(grid_search.best_params_, np.sqrt(-grid_search.best_score_))
        grid_search.cv_results_['mean_test_score'] = np.sqrt(-grid_search.cv_results_['mean_test_score'])
        print(pd.DataFrame(grid_search.cv_results_)[['params','mean_test_score','std_test_score']])

### parameter
- xgb
  - 'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 1.0

# ensemble

In [ ]:
class AverageWeight(BaseEstimator, RegressorMixin):
    def __init__(self,mod,weight):
        self.mod = mod
        self.weight = weight
        
    def fit(self,X,y):
        self.models_ = [clone(x) for x in self.mod]
        for model in self.models_:
            model.fit(X,y)
        return self
    
    def predict(self,X):
        w = list()
        pred = np.array([model.predict(X) for model in self.models_])
        # for every data point, single model prediction times weight, then add them together
        for data in range(pred.shape[1]):
            single = [pred[model,data]*weight for model,weight in zip(range(pred.shape[0]),self.weight)]
            w.append(np.sum(single))
        return w

In [ ]:
xgb=XGBRegressor(colsample_bytree= 0.6, learning_rate= 0.1, max_depth= 8, min_child_weight= 1, subsample= 1.0)
rf=RandomForestRegressor(610,n_jobs=-1)
extra=ExtraTreesRegressor(n_jobs=-1)

In [ ]:
# assign weights based on their gridsearch score
w1 = 0.3

w2 = 0.35
w3 = 0.35



In [ ]:
weight_avg = AverageWeight(mod = [rf,xgb,extra],weight=[w1,w2,w3])

## score 

In [ ]:
score =rmse_cv(weight_avg,full_selected,price_res['單價元平方公尺']),  rmse_cv(weight_avg,full_selected,price_res['單價元平方公尺']).mean()
score

## fit

In [ ]:
weight_avg.fit(full_selected,price_res['單價元平方公尺'])


In [ ]:

# Apply the pipeline to your data
test = test.copy()
test_transformed = pipeline.fit_transform(test)
test_selected = test_transformed.drop(features_to_drop, axis=1)
#print(test_transformed)

In [ ]:
res=weight_avg.predict(test_selected)

In [ ]:

 
# 建立 DataFrame
df = pd.DataFrame({'單價元平方公尺': res})

# 新增 Id 欄位
# Create a DataFrame, prioritizing 'Id'
df = pd.DataFrame({'Id': df.index, '單價元平方公尺': res})

# Specify the output directory
output_dir = './output'  # Replace with your desired path
output_file = output_dir + '/output.csv'



# # 將 DataFrame 輸出為 CSV 檔案
df.to_csv(output_file, index=False)